# 00 · 环境自检与数据构建

**目标**：确认 T4 GPU、钉死兼容参数、构建三任务数据集并落 Drive。

**硬件**：T4 (16GB, sm7.5)。本 notebook 不训练模型。

In [ ]:
# [0] GPU 自检：确认拿到 T4，并钉死 T4 兼容参数
import torch
assert torch.cuda.is_available(), "未分配到 GPU！请 运行时->更改运行时类型 选 T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} | VRAM: {vram:.1f} GB | CC: {torch.cuda.get_device_capability(0)}")
# T4 = sm7.5：无原生 bf16、无 FlashAttention-2 -> 统一 fp16 + sdpa
DTYPE, ATTN = "float16", "sdpa"
print("钉死参数: dtype=%s attn=%s (T4 无 bf16/FA2)" % (DTYPE, ATTN))


In [ ]:
# [1] 依赖对齐
# 关键：不钉 torch（用 Colab 预装 torch，钉旧版会因下架而失败，见 pitfalls P-11）
# 钉死 transformers/trl 兼容矩阵（trl 0.x 与 transformers 5.x 不兼容，见 P-04）
!pip install -q "transformers==4.57.3" "trl==0.15.0" "peft" "accelerate" "bitsandbytes" \
    "datasets" "sentencepiece" "tiktoken" "pandas" "matplotlib"
import torch, transformers, trl
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, "| trl", trl.__version__)


In [ ]:
# [2] 克隆仓库并把 src 加入 sys.path（单一事实源）
# 注意：Colab 上 `pip install -e .`(PEP660 editable) 常出现装完仍 import 不到的坑，
# 故不用 editable，直接 sys.path 注入 src（依赖已在 [1] 装好）。
import os, subprocess, sys
if not os.path.exists("cs0502-mini-llm"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Zhe-SH-CN/cs0502-mini-llm.git"], check=True)
_src = os.path.abspath("cs0502-mini-llm/src")
if _src not in sys.path:
    sys.path.insert(0, _src)
import minillm
from minillm.config import ExpConfig
from minillm import data as D
from minillm import evaluate as E
from minillm import train as T
from minillm import quantize as Q
from minillm import bench as B
from minillm.record import Recorder
print("minillm", minillm.__version__, "| src:", _src)


In [ ]:
# [3] 挂载 Google Drive，统一结果/数据落盘路径（Colab 文件系统易失，必须落 Drive）
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/cs0502-mini-llm"
DATA = os.path.join(BASE, "data"); RESULTS = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
rec = Recorder(RESULTS)
print("DATA:", DATA, "| RESULTS:", RESULTS)


In [ ]:
# [4] 构建三任务数据集（确定性，seed 固定）并做分层切分（courseqa 按 topic 分层防泄漏）
import json
stats = D.build_all_splits(DATA, seed=42)
print(json.dumps(stats, ensure_ascii=False, indent=1))


In [ ]:
# [5] 抽样检查数据质量（人工 sanity check）
from minillm import data as D
import os
for task in ("courseqa", "jsonout", "intent"):
    ds = D.load_dataset(task, DATA)
    print("\n###", task, ds.sizes())
    for r in ds.split("test")[:3]:
        print(" ", json.dumps(r, ensure_ascii=False)[:220])


## 完成

数据已落 `Drive/cs0502-mini-llm/data/`。下一步运行 `01_minimind_sft_eval.ipynb`。